# 90-Day Repeat-Purchase Propensity Dataset

This notebook constructs a leakage-safe customer-level dataset for predicting
whether an existing customer will make another valid merchandise purchase
during the following 90 days.

Unlike the full-history segmentation analysis, propensity modelling requires
a strict separation between historical information used as predictors and
future transactions used to construct the target.

All customer features are therefore calculated using transactions available
at or before a fixed historical cutoff date. Transactions after the cutoff date are not
used in feature engineering and are reserved exclusively for determining
whether the customer purchases again during the 90-day prediction window.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import requests


REPO_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

In [2]:
LOCAL_DATA_PATH = (
    REPO_ROOT
    / "data"
    / "processed"
    / "online_retail_clean.parquet"
)

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "apostolis-bloutsos-data/"
    "customer-segmentation-purchase-propensity/"
    "main/data/processed/online_retail_clean.parquet"
)

if LOCAL_DATA_PATH.exists():
    df = pd.read_parquet(LOCAL_DATA_PATH)
    print("Loaded local online retail clean dataset.")

else:
    response = requests.get(DATA_URL, timeout=120)
    response.raise_for_status()

    LOCAL_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    LOCAL_DATA_PATH.write_bytes(response.content)

    df = pd.read_parquet(LOCAL_DATA_PATH)
    print("Downloaded cleaned dataset from GitHub.")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Downloaded cleaned dataset from GitHub.
Rows: 820,506
Columns: 14


In [3]:
purchases = df[
    df["transaction_type"] == "purchase"
].copy()

cancellations = df[
    df["transaction_type"] == "cancellation"
].copy()

print("First purchase:", purchases["invoice_date"].min())
print("Last purchase: ", purchases["invoice_date"].max())

First purchase: 2009-12-01 07:45:00
Last purchase:  2011-12-09 12:50:00


In [4]:
CUTOFF_DATE = pd.Timestamp("2011-09-09 23:59:59")
TARGET_END_DATE = CUTOFF_DATE + pd.Timedelta(days=90)

print("Historical cutoff:", CUTOFF_DATE)
print("Target window end:", TARGET_END_DATE)

Historical cutoff: 2011-09-09 23:59:59
Target window end: 2011-12-08 23:59:59


## Temporal design

The historical cutoff is set to the end of 9 September 2011.

Customer predictors are constructed exclusively from transactions occurring on
or before this cutoff date. The subsequent 90 calendar days, from 10 September
through 8 December 2011, are kept for target construction.

The final day of available data is intentionally left outside the target window,
ensuring that the complete 90-day prediction horizon is observable.

A positive target (`repeat_purchase_90d = 1`) means that an eligible customer
makes at least one valid merchandise purchase during this future 90-day window.
A negative target (`repeat_purchase_90d = 0`) means that such a purchase is not
observed during the same period.

Cancellations do not count as repeat purchases.

In [5]:
historical_df = df[
    df["invoice_date"] <= CUTOFF_DATE
].copy()

target_df = df[
    (df["invoice_date"] > CUTOFF_DATE)
    & (df["invoice_date"] <= TARGET_END_DATE)
].copy()

print(f"Historical rows: {len(historical_df):,}")
print(f"Target-window rows: {len(target_df):,}")

Historical rows: 656,582
Target-window rows: 163,306


In [6]:
historical_purchases = historical_df[
    historical_df["transaction_type"] == "purchase"
].copy()

historical_cancellations = historical_df[
    historical_df["transaction_type"] == "cancellation"
].copy()

future_purchases = target_df[
    target_df["transaction_type"] == "purchase"
].copy()

In [7]:
print(
    "Historical purchase range:",
    historical_purchases["invoice_date"].min(),
    "to",
    historical_purchases["invoice_date"].max()
)

print(
    "Future purchase range:",
    future_purchases["invoice_date"].min(),
    "to",
    future_purchases["invoice_date"].max()
)

Historical purchase range: 2009-12-01 07:45:00 to 2011-09-09 15:53:00
Future purchase range: 2011-09-11 10:35:00 to 2011-12-08 20:01:00


In [8]:
eligible_customers = (
    historical_purchases["customer_id"]
    .drop_duplicates()
)

print(
    f"Eligible customers at cutoff: "
    f"{len(eligible_customers):,}"
)

Eligible customers at cutoff: 5,256


In [9]:
future_repeat_customers = set(
    future_purchases["customer_id"]
)

repeat_count = eligible_customers.isin(
    future_repeat_customers
).sum()

print(f"Repeat purchasers within 90 days: {repeat_count:,}")
print(
    f"Repeat-purchase rate: "
    f"{repeat_count / len(eligible_customers) * 100:.2f}%"
)

Repeat purchasers within 90 days: 2,287
Repeat-purchase rate: 43.51%


## Target construction and historical customer features

The modelling population consists of customers who had made at least one valid
merchandise purchase by the historical cutoff.

For each eligible customer, the binary target is defined as:

- `1` — at least one valid merchandise purchase is observed during the following
  90-day target window;
- `0` — no valid merchandise purchase is observed during that window.

The target is reasonably balanced, with approximately 43.5% of eligible
customers purchasing again within 90 days. The original class distribution is
therefore retained without artificial over- or under-sampling.

All predictor variables are calculated exclusively from transactions occurring
on or before the cutoff date.

In [10]:
propensity_df = pd.DataFrame({
    "customer_id": eligible_customers
})

In [11]:
propensity_df["repeat_purchase_90d"] = (
    propensity_df["customer_id"]
    .isin(future_repeat_customers)
    .astype(int)
)

In [12]:
print(propensity_df["repeat_purchase_90d"].value_counts())
print()
print(
    propensity_df["repeat_purchase_90d"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

repeat_purchase_90d
0    2969
1    2287
Name: count, dtype: int64

repeat_purchase_90d
0    56.49
1    43.51
Name: proportion, dtype: float64


In [13]:
assert len(propensity_df) == len(eligible_customers)
assert propensity_df["customer_id"].is_unique
assert set(
    propensity_df["repeat_purchase_90d"].unique()
) == {0, 1}

print("Target construction passed.")

Target construction passed.


### Historical purchasing behaviour

Lifetime-to-cutoff purchase behaviour is summarized using recency, purchase
frequency, merchandise value, purchased quantity, product breadth and observed
customer tenure.

All measures are calculated strictly from purchases observed on or before the
historical cutoff.

In [14]:
historical_features = (
    historical_purchases
    .groupby("customer_id")
    .agg(
        first_purchase_date=("invoice_date", "min"),
        last_purchase_date=("invoice_date", "max"),
        frequency=("invoice", "nunique"),
        monetary_value=("line_revenue", "sum"),
        total_items=("quantity", "sum"),
        unique_products=("stock_code", "nunique")
    )
    .reset_index()
)

In [15]:
historical_features["recency_days"] = (
    CUTOFF_DATE
    - historical_features["last_purchase_date"]
).dt.days

In [16]:
historical_features["tenure_days"] = (
    CUTOFF_DATE
    - historical_features["first_purchase_date"]
).dt.days

In [17]:
historical_features["average_order_value"] = (
    historical_features["monetary_value"]
    / historical_features["frequency"]
)

### Recent purchase momentum

Two customers with the same historical purchase frequency may have very
different current behaviour. For example, a customer who purchased ten times
historically but has been inactive for the past year differs substantially from
a customer whose ten purchases were concentrated recently.

A 90-day historical activity window ending at the cutoff is therefore used to
capture recent purchase momentum. Four features are constructed from this
period: purchase frequency, merchandise value, purchased quantity and product
breadth.

In [18]:
HISTORY_90D_START = (
    CUTOFF_DATE - pd.Timedelta(days=90)
)

recent_90d_purchases = historical_purchases[
    historical_purchases["invoice_date"] > HISTORY_90D_START
].copy()

In [19]:
recent_features = (
    recent_90d_purchases
    .groupby("customer_id")
    .agg(
        purchases_last_90d=("invoice", "nunique"),
        monetary_last_90d=("line_revenue", "sum"),
        items_last_90d=("quantity", "sum"),
        unique_products_last_90d=("stock_code", "nunique")
    )
    .reset_index()
)

### Historical cancellation behaviour

Historical cancellation behaviour is also summarized using only transactions
observed on or before the cutoff date. Cancellation count, cancellation value
and cancellation value relative to historical merchandise value are retained
as potential predictors.

In [20]:
historical_cancellation_features = (
    historical_cancellations
    .groupby("customer_id")
    .agg(
        cancellation_count=("invoice", "nunique"),
        cancellation_value=(
            "line_revenue",
            lambda x: -x.sum()
        )
    )
    .reset_index()
)

### Assemble the customer-level modelling table

The target, historical purchasing features, recent-activity measures and
historical cancellation features are merged into a single customer-level
dataset.

In [21]:
propensity_df = (
    propensity_df
    .merge(
        historical_features,
        on="customer_id",
        how="left"
    )
    .merge(
        recent_features,
        on="customer_id",
        how="left"
    )
    .merge(
        historical_cancellation_features,
        on="customer_id",
        how="left"
    )
)

Customers without purchase or cancellation activity in the relevant historical
window receive zero for the corresponding behavioural features rather than a
missing value. In this context, zero represents the observed absence of the
behaviour and is therefore greatly different from unknown or missing data.

In [22]:
zero_fill_columns = [
    "purchases_last_90d",
    "monetary_last_90d",
    "items_last_90d",
    "unique_products_last_90d",
    "cancellation_count",
    "cancellation_value"
]

propensity_df[zero_fill_columns] = (
    propensity_df[zero_fill_columns]
    .fillna(0)
)

In [23]:
propensity_df["cancellation_value_rate"] = (
    propensity_df["cancellation_value"]
    / propensity_df["monetary_value"]
)

In [24]:
assert (
    historical_purchases["invoice_date"].max()
    <= CUTOFF_DATE
)

assert (
    historical_cancellations["invoice_date"].max()
    <= CUTOFF_DATE
)

assert (
    future_purchases["invoice_date"].min()
    > CUTOFF_DATE
)

assert len(propensity_df) == len(eligible_customers)
assert propensity_df["customer_id"].is_unique

print("Temporal leakage checks passed.")

Temporal leakage checks passed.


In [25]:
propensity_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5256 entries, 0 to 5255
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   customer_id               5256 non-null   string        
 1   repeat_purchase_90d       5256 non-null   int64         
 2   first_purchase_date       5256 non-null   datetime64[ns]
 3   last_purchase_date        5256 non-null   datetime64[ns]
 4   frequency                 5256 non-null   int64         
 5   monetary_value            5256 non-null   float64       
 6   total_items               5256 non-null   int64         
 7   unique_products           5256 non-null   int64         
 8   recency_days              5256 non-null   int64         
 9   tenure_days               5256 non-null   int64         
 10  average_order_value       5256 non-null   float64       
 11  purchases_last_90d        5256 non-null   float64       
 12  monetary_last_90d   

In [26]:
propensity_df.head()

,customer_id,repeat_purchase_90d,first_purchase_date,last_purchase_date,frequency,monetary_value,total_items,unique_products,recency_days,tenure_days,average_order_value,purchases_last_90d,monetary_last_90d,items_last_90d,unique_products_last_90d,cancellation_count,cancellation_value,cancellation_value_rate
0,13085,0,2009-12-01 07:45:00,2011-07-05 12:11:00,8,2433.28,944,50,66,647,304.160000,1.0,137.98,44.0,9.0,1.0,143.70,0.059056
1,13078,1,2009-12-01 09:06:00,2011-08-25 10:16:00,46,23821.66,9053,141,15,647,517.862174,5.0,2449.99,1060.0,52.0,33.0,597.54,0.025084
2,15362,0,2009-12-01 09:08:00,2010-09-17 10:37:00,2,613.08,368,38,357,647,306.540000,0.0,0.00,0.0,0.0,0.0,0.00,0.000000
3,18102,1,2009-12-01 09:24:00,2011-09-02 11:53:00,112,484615.10,155689,311,7,647,4326.920536,10.0,43321.12,10504.0,17.0,2.0,2578.40,0.005321
4,12682,1,2009-12-01 09:28:00,2011-08-31 12:18:00,40,18039.61,8867,280,9,647,450.990250,9.0,2654.63,1291.0,78.0,1.0,88.10,0.004884


In [27]:
predictor_columns = [
    "recency_days",
    "frequency",
    "monetary_value",
    "total_items",
    "unique_products",
    "average_order_value",
    "tenure_days",
    "purchases_last_90d",
    "monetary_last_90d",
    "items_last_90d",
    "unique_products_last_90d",
    "cancellation_count",
    "cancellation_value",
    "cancellation_value_rate"
]

In [28]:
# Check predictors for missing or non-finite values.

print("Missing values:")
print(propensity_df[predictor_columns].isna().sum())

assert propensity_df[predictor_columns].notna().all().all()
assert np.isfinite(
    propensity_df[predictor_columns].to_numpy()
).all()

print("No missing or non-finite predictor values.")

Missing values:
recency_days                0
frequency                   0
monetary_value              0
total_items                 0
unique_products             0
average_order_value         0
tenure_days                 0
purchases_last_90d          0
monetary_last_90d           0
items_last_90d              0
unique_products_last_90d    0
cancellation_count          0
cancellation_value          0
cancellation_value_rate     0
dtype: int64
No missing or non-finite predictor values.


In [29]:
# Validate logical relationships among predictors.

assert (propensity_df["recency_days"] >= 0).all()
assert (propensity_df["tenure_days"] >= propensity_df["recency_days"]).all()

assert (propensity_df["frequency"] >= 1).all()
assert (propensity_df["monetary_value"] > 0).all()
assert (propensity_df["total_items"] > 0).all()
assert (propensity_df["unique_products"] >= 1).all()
assert (propensity_df["average_order_value"] > 0).all()

assert (
    propensity_df["purchases_last_90d"]
    <= propensity_df["frequency"]
).all()

assert (
    propensity_df["monetary_last_90d"]
    <= propensity_df["monetary_value"]
).all()

assert (
    propensity_df["items_last_90d"]
    <= propensity_df["total_items"]
).all()

assert (
    propensity_df["unique_products_last_90d"]
    <= propensity_df["unique_products"]
).all()

assert (propensity_df["cancellation_count"] >= 0).all()
assert (propensity_df["cancellation_value"] >= 0).all()
assert (propensity_df["cancellation_value_rate"] >= 0).all()

print("Predictor consistency checks passed.")

Predictor consistency checks passed.


In [30]:
propensity_df[predictor_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
recency_days,5256.0,205.887747,174.372666,0.00,49.000000,163.000,324.000000,647.00000
frequency,5256.0,5.711568,11.213954,1.00,1.000000,3.000,6.000000,284.00000
monetary_value,5256.0,2672.149019,12407.311827,1.55,322.287500,797.905,2106.925000,484615.10000
total_items,5256.0,1662.778919,8152.418649,1.00,175.000000,451.500,1244.250000,297934.00000
unique_products,5256.0,74.552131,104.980614,1.00,18.000000,41.000,92.000000,2178.00000
average_order_value,5256.0,376.622732,596.276992,1.55,178.633125,282.309,417.980000,19338.49000
tenure_days,5256.0,430.685312,180.702939,0.00,310.000000,473.000,586.000000,647.00000
purchases_last_90d,5256.0,0.750000,1.792229,0.00,0.000000,0.000,1.000000,45.00000
monetary_last_90d,5256.0,348.729131,1947.021846,0.00,0.000000,0.000,268.857500,62151.70000
items_last_90d,5256.0,216.968037,1187.104042,0.00,0.000000,0.000,146.000000,46111.00000


In [31]:
predictor_skewness = (
    propensity_df[predictor_columns]
    .skew()
    .sort_values(ascending=False)
)

predictor_skewness

,0
cancellation_value,51.095571
monetary_value,24.237213
items_last_90d,22.374621
monetary_last_90d,20.416343
total_items,20.229971
average_order_value,16.370469
cancellation_value_rate,14.563432
frequency,10.489872
cancellation_count,9.707584
purchases_last_90d,9.047421


## Predictor distribution observations

The historical customer predictors show substantial heterogeneity and strong
right-skew across most purchasing and cancellation variables.

Long-term purchase activity is highly concentrated. The median customer has
made only **3 purchases** with approximately **£798** in historical merchandise
value, while the most active and valuable customers reach **284 purchase
invoices** and more than **£484,000** in merchandise value. Similar long-tailed
patterns are observed for purchased quantity, product breadth and average order
value.

The recent 90-day activity variables are particularly sparse. Their medians are
zero, indicating that at least half of eligible customers made no purchase
during the 90 days immediately preceding the cutoff. A smaller group of highly
active customers generates very large recent purchase counts, values and
quantities. These features therefore capture a strong distinction between
recently active and inactive customers, which may be useful for predicting
near-term repeat purchasing.

Cancellation behaviour is also highly zero-inflated. The median customer has no
recorded historical cancellation activity, while a relatively small number of
customers exhibit much larger cancellation counts and values.

The skewness statistics confirm these patterns. Cancellation value, monetary
value, recent monetary activity, purchased quantities, average order value and
purchase frequency are all strongly right-skewed. In contrast, recency shows
only moderate positive skew, while observed tenure is mildly left-skewed.

These findings do not by themselves indicate erroneous observations. Large
values may represent genuine high-activity or high-value customers. Predictor
transformations and scaling will therefore be considered during model
development rather than removing extreme customers mechanically.

Importantly, any transformations will be fitted only after the training/test
split in the modelling notebook so that information from the held-out test
population does not influence model preparation.

In [32]:
# Final target distribution validation.
target_summary = (
    propensity_df["repeat_purchase_90d"]
    .value_counts()
    .sort_index()
    .rename_axis("repeat_purchase_90d")
    .to_frame("customers")
)

target_summary["pct"] = (
    target_summary["customers"]
    / len(propensity_df)
    * 100
)

target_summary

,customers,pct
repeat_purchase_90d,,
0,2969,56.487823
1,2287,43.512177


In [33]:
PROPENSITY_DATA_PATH = (
    REPO_ROOT
    / "data"
    / "processed"
    / "propensity_dataset.parquet"
)

PROPENSITY_DATA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

propensity_df.to_parquet(
    PROPENSITY_DATA_PATH,
    index=False
)

print(f"Saved to: {PROPENSITY_DATA_PATH}")

Saved to: /content/data/processed/propensity_dataset.parquet


In [34]:
test_propensity = pd.read_parquet(PROPENSITY_DATA_PATH)

assert len(test_propensity) == len(propensity_df)
assert list(test_propensity.columns) == list(propensity_df.columns)
assert test_propensity["customer_id"].is_unique

print("Propensity dataset Parquet round-trip validation passed.")

Propensity dataset Parquet round-trip validation passed.


## Final propensity modelling dataset

The final dataset contains one row for each customer who had made at least one
valid merchandise purchase by the historical cutoff.

All predictor variables are derived exclusively from information available on
or before 9 September 2011. The subsequent 90-day period is used only to
construct the binary repeat-purchase target.

The feature set combines three complementary views of historical behaviour:

- **long-term purchasing behaviour**, including recency, frequency, customer
  value, quantity, product breadth, order value and observed tenure;
- **recent purchase momentum**, measured over the 90 days immediately preceding
  the cutoff; and
- **historical cancellation behaviour**.

The resulting dataset contains **5,256 eligible customers**, of whom **2,287
(43.51%)** make another valid merchandise purchase within the following
90 days and **2,969 (56.49%)** do not.

Customer identifiers and the derived first- and last-purchase timestamps are
retained for traceability but are not intended to be used directly as model
predictors. Feature transformations, scaling and any target-dependent
exploratory analysis are intentionally deferred until after the training/test
split in the modelling notebook, preventing information from the held-out test
population from influencing model-development decisions.